In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType

try:
    # 1. LEER la geografía real de forma directa y eficiente
    df_geo = spark.read.table("lk_costarica_clima.dbo.bronce_geo_cantones")
    
    # 2. GENERAR Y TRANSFORMAR en un solo paso optimizado (Sin usar RDD)
    # Aplicamos el CAST directo a DECIMAL para evitar decimales infinitos y ahorrar memoria
    df_clima_final = df_geo.select("canton_id").distinct() \
        .withColumn("fecha_registro", F.current_timestamp()) \
        .withColumn("temperatura_celsius", F.round((F.rand() * 15) + 18, 1).cast(DecimalType(4,1))) \
        .withColumn("precipitacion_mm", F.round(F.rand() * 50, 2).cast(DecimalType(5,2))) \
        .withColumn("humedad_porcentaje", F.round((F.rand() * 40) + 60, 0).cast(IntegerType()))

    # 3. GUARDAR directamente en OneLake (Capa Bronce)
    # Al usar funciones nativas, Spark ejecuta esto en milisegundos consumiendo casi 0% de capacidad
    df_clima_final.write.format("delta").mode("append").saveAsTable("lk_costarica_clima.dbo.bronce_meteo_lecturas")
    
    print("¡Éxito total! Datos climáticos optimizados y guardados en la capa Bronce.")
    df_clima_final.show(5)

except Exception as e:
    print(f"Error en el pipeline de clima optimizado: {e}")


StatementMeta(, e18b9c0c-6f19-43a8-acd2-f2a2c8feb3af, 19, Finished, Available, Finished, False)

¡Éxito rotundo! Datos de sensores climáticos generados y acumulados en la capa Bronce.
+---------+--------------------+-------------------+----------------+------------------+
|canton_id|      fecha_registro|temperatura_celsius|precipitacion_mm|humedad_porcentaje|
+---------+--------------------+-------------------+----------------+------------------+
|      613|2026-09-20 22:22:...|               31.9|            32.6|                97|
|      606|2026-09-20 22:22:...|               32.8|            11.3|                88|
|      704|2026-09-20 22:22:...|               26.6|            31.2|                98|
|      607|2026-09-20 22:22:...|               27.7|            43.9|                85|
|      609|2026-09-20 22:22:...|               29.7|             7.7|                70|
+---------+--------------------+-------------------+----------------+------------------+
only showing top 5 rows



In [4]:
df_geo = spark.read.table("bronce_geo_cantones")

print(df_geo.count())

StatementMeta(, e18b9c0c-6f19-43a8-acd2-f2a2c8feb3af, 8, Finished, Available, Finished, False)

84
